In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

TRAIN_DATASET_DIR = "./learn_dataset_single_object"  # 学習: 単一物体
TEST_DATASET_DIR  = "./learn_dataset_fixed_angle"     # テスト: 2物体同時
TRAIN_META_CSV = os.path.join(TRAIN_DATASET_DIR, "metadata.csv")
TEST_META_CSV  = os.path.join(TEST_DATASET_DIR,  "metadata.csv")
MODEL_PATH = "./best_detector_agnostic.pt"
#MODEL_PATH = "./sweep_models/best_sr5.0_a2_b8.pt"
N_FIXED      = 10
FIXED_ANGLES = np.linspace(-5, 5, N_FIXED)  # 角度グリッド [度]
SIGMA_ANGLE  = 0.3                           # 角度方向Gaussianの標準偏差 [度]

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 8
EPOCHS     = 30
LR         = 1e-4

D_TOL = 2
R_TOL = 3

print("DEVICE:", DEVICE)
print("TRAIN_META_CSV exists:", os.path.exists(TRAIN_META_CSV))
print("TEST_META_CSV  exists:", os.path.exists(TEST_META_CSV))
print(torch.cuda.is_available())

print(torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = False
torch.cuda.empty_cache()
print(f"allocated: {torch.cuda.memory_allocated()/1024**2:.0f} MB")
print(f"reserved:  {torch.cuda.memory_reserved()/1024**2:.0f} MB")

DEVICE: cuda
TRAIN_META_CSV exists: True
TEST_META_CSV  exists: True
True
NVIDIA GeForce RTX 3060 Ti
allocated: 0 MB
reserved:  0 MB


In [2]:
class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels=32, dropout=0.1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout3d(dropout),
        )

    def forward(self, x):
        return self.block(x)


In [9]:
# True: 単一物体のみで学習（汎化実験）/ False: 単一+2物体の混合学習（通常）
GENERALIZATION_EXPERIMENT = True

RANDOM_SEED = 42

# --- 単一物体データ ---
single_df = pd.read_csv(TRAIN_META_CSV)
single_df = single_df[single_df["valid_all"] == 1].reset_index(drop=True)
single_df = single_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
single_train_df = single_df.iloc[:280].reset_index(drop=True)
single_val_df   = single_df.iloc[280:340].reset_index(drop=True)

# --- 2物体データ ---
two_df = pd.read_csv(TEST_META_CSV)
two_df = two_df[two_df["valid_all"] == 1].reset_index(drop=True)
two_df = two_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
two_train_df = two_df.iloc[:170].reset_index(drop=True)
two_val_df   = two_df.iloc[170:200].reset_index(drop=True)
test_df      = two_df.iloc[200:].reset_index(drop=True)

# --- フラグで切り替え ---
if GENERALIZATION_EXPERIMENT:
    train_df       = single_train_df
    val_df         = single_val_df
    MODEL_PATH_SEG = "./best_detector_softmax_heatmap_single_train.pt"
else:
    train_df       = pd.concat([single_train_df, two_train_df], ignore_index=True)
    val_df         = pd.concat([single_val_df,   two_val_df],   ignore_index=True)
    MODEL_PATH_SEG = "./best_detector_softmax_heatmap.pt"

print(f"GENERALIZATION_EXPERIMENT: {GENERALIZATION_EXPERIMENT}")
print(f"train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")
print(f"MODEL_PATH_SEG: {MODEL_PATH_SEG}")

GENERALIZATION_EXPERIMENT: True
train: 280, val: 60, test: 100
MODEL_PATH_SEG: ./best_detector_softmax_heatmap_single_train.pt


In [4]:
def load_all_rd_maps(npz_path):
    data = np.load(npz_path)
    rd_maps = data["rd_maps"]
    result = []
    for i in range(rd_maps.shape[0]):
        rd_mag = np.abs(rd_maps[i]).astype(np.float32)
        rd_db = 20.0 * np.log10(np.maximum(rd_mag, 1e-12))
        result.append(rd_db)
    return np.stack(result, axis=0)


In [5]:
# ===== softmaxヒートマップモデル =====
# 3クラス(背景/cy/ve)をボクセルごとにsoftmaxで分類する
# exist_headを廃止し、確率崩壊を構造的に解消する

class RadarUNet3DSoftmax(nn.Module):
    """
    3D UNet backboneはそのままに、出力ヘッドを3クラスに変更。
    各ボクセルが (背景=0 / cyclist=1 / vehicle=2) のlogitを持つ。
    閾値ベースのピーク検出で検出/未検出を判断するためexist_headは不要。
    """
    def __init__(self, n_angles=N_FIXED, fixed_channels=32, dropout=0.1):
        super().__init__()
        self.n_angles = n_angles

        # 入力: (B, 1, N_FIXED, H, W)
        self.encoders = nn.ModuleList([
            ConvBlock3D(1,              fixed_channels, dropout),
            ConvBlock3D(fixed_channels, fixed_channels, dropout),
            ConvBlock3D(fixed_channels, fixed_channels, dropout),
            ConvBlock3D(fixed_channels, fixed_channels, dropout),
        ])
        # 角度軸はプールしない（N_FIXED=10が小さいため）
        self.pools = nn.ModuleList([
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2)) for _ in range(4)
        ])
        self.bottleneck = ConvBlock3D(fixed_channels, fixed_channels, dropout)
        self.upsamples = nn.ModuleList([
            nn.Upsample(scale_factor=(1, 2, 2), mode="trilinear", align_corners=False)
            for _ in range(4)
        ])
        self.decoders = nn.ModuleList([
            ConvBlock3D(fixed_channels * 2, fixed_channels, dropout)
            for _ in range(4)
        ])
        # 3クラス出力ヘッド: softmaxはF.cross_entropy内部で処理
        self.seg_head = nn.Conv3d(fixed_channels, 3, kernel_size=1)

    def forward(self, x):
        # x: (B, N_FIXED, H, W) → (B, 1, N_FIXED, H, W)
        x_3d = x.unsqueeze(1)

        skips = []
        feat = x_3d
        for enc, pool in zip(self.encoders, self.pools):
            feat = enc(feat)
            skips.append(feat)
            feat = pool(feat)
        feat = self.bottleneck(feat)
        for up, dec, skip in zip(self.upsamples, self.decoders, reversed(skips)):
            feat = up(feat)
            if feat.shape[-3:] != skip.shape[-3:]:
                feat = F.interpolate(feat, size=skip.shape[-3:], mode="trilinear", align_corners=False)
            feat = torch.cat([feat, skip], dim=1)
            feat = dec(feat)

        return self.seg_head(feat)  # (B, 3, N_FIXED, H, W)

In [6]:
# ===== 点ラベルデータセット =====
# 正解ラベルを点(0/1/2の離散値)で与える
# Gaussianは確率崩壊回避のための回避策だったが、softmaxで不要になる

class MultiTargetDatasetSeg(Dataset):
    def __init__(self, meta_df):
        self.meta_df = meta_df.reset_index(drop=True)
        self.samples = []
        print(f"データ読み込み中... ({len(self.meta_df)} サンプル)", flush=True)
        for idx in range(len(self.meta_df)):
            row = self.meta_df.iloc[idx]
            npz_path = row["file"]
            if not os.path.isabs(npz_path):
                npz_path = os.path.normpath(os.path.join(".", npz_path))

            data = np.load(npz_path)
            rd_maps = data["rd_maps"]
            x = np.stack(
                [20.0 * np.log10(np.maximum(np.abs(rd_maps[i]).astype(np.float32), 1e-12))
                 for i in range(rd_maps.shape[0])],
                axis=0
            )  # (N_FIXED, H, W)
            H, W = x.shape[1], x.shape[2]

            fixed_angles = data["fixed_angles"] if "fixed_angles" in data else FIXED_ANGLES

            valid_cy = 1 if str(row["valid_cyclist"]).strip() in ("1", "True") else 0
            valid_ve = 1 if str(row["valid_vehicle"]).strip() in ("1", "True") else 0

            cy_angle = float(data["cyclist_true_angle_deg"]) if (valid_cy and "cyclist_true_angle_deg" in data) else 0.0
            ve_angle = float(data["vehicle_true_angle_deg"]) if (valid_ve and "vehicle_true_angle_deg" in data) else 0.0

            # 点ラベル: 0=背景 / 1=cy / 2=ve
            # cy/veが同じボクセルに重なった場合はveが優先（上書き）
            y_seg = np.zeros((N_FIXED, H, W), dtype=np.int64)
            if valid_cy:
                cy_ch = int(np.argmin(np.abs(fixed_angles - cy_angle)))
                y_seg[cy_ch, int(row["cyclist_true_d_idx"]), int(row["cyclist_true_r_idx"])] = 1
            if valid_ve:
                ve_ch = int(np.argmin(np.abs(fixed_angles - ve_angle)))
                y_seg[ve_ch, int(row["vehicle_true_d_idx"]), int(row["vehicle_true_r_idx"])] = 2

            self.samples.append({
                "x":         torch.from_numpy(x).float(),
                "y_seg":     torch.from_numpy(y_seg).long(),  # (N_FIXED, H, W)
                "cy_true_d": torch.tensor(int(row["cyclist_true_d_idx"]), dtype=torch.long),
                "cy_true_r": torch.tensor(int(row["cyclist_true_r_idx"]), dtype=torch.long),
                "ve_true_d": torch.tensor(int(row["vehicle_true_d_idx"]), dtype=torch.long),
                "ve_true_r": torch.tensor(int(row["vehicle_true_r_idx"]), dtype=torch.long),
                "valid_cy":  torch.tensor(valid_cy, dtype=torch.long),
                "valid_ve":  torch.tensor(valid_ve, dtype=torch.long),
            })
        print("読み込み完了", flush=True)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
# ===== softmax用 損失・デコード・学習ループ =====

# 正例1ピクセル vs 背景~16万ピクセルの不均衡をクラス重みで補正
SEG_BG_WEIGHT = 1.0
SEG_CY_WEIGHT = 500.0
SEG_VE_WEIGHT = 500.0

def compute_seg_loss(logits, y_seg):
    # logits: (B, 3, N_FIXED, H, W)
    # y_seg:  (B, N_FIXED, H, W) long, 値は{0,1,2}
    weight = torch.tensor([SEG_BG_WEIGHT, SEG_CY_WEIGHT, SEG_VE_WEIGHT],
                          dtype=torch.float32, device=logits.device)
    return F.cross_entropy(logits, y_seg, weight=weight)


def decode_softmax_detections(logits, threshold=0.5):
    """
    softmax確率マップから閾値を超えるピークをすべて列挙して返す。
    NMSで同一ターゲット由来の近傍ピークを抑制する。
    戻り値: (cy_detections, ve_detections)
      各要素は [(ch, d, r), ...] のリスト。未検出なら空リスト。
    """
    probs = F.softmax(logits, dim=1)   # (1, 3, N_FIXED, H, W)
    cy_prob = probs[0, 1]              # (N_FIXED, H, W)
    ve_prob = probs[0, 2]              # (N_FIXED, H, W)

    def _detect_all(prob_map, thr):
        # NMSで閾値超えピークを順に取り出す（旧3D NMSに合わせて ch±1, d±3, r±1 を抑制）
        prob = prob_map.clone()
        N, H, W = prob.shape
        detections = []
        while prob.max().item() >= thr:
            flat_idx = torch.argmax(prob).item()
            ch  = flat_idx // (H * W)
            rem = flat_idx %  (H * W)
            d   = rem // W
            r   = rem %  W
            detections.append((ch, d, r))
            ch_lo, ch_hi = max(ch-1, 0), min(ch+1, N-1)
            d_lo,  d_hi  = max(d-3,  0), min(d+3,  H-1)
            r_lo,  r_hi  = max(r-1,  0), min(r+1,  W-1)
            prob[ch_lo:ch_hi+1, d_lo:d_hi+1, r_lo:r_hi+1] = 0.0
        return detections

    return _detect_all(cy_prob, threshold), _detect_all(ve_prob, threshold)


def run_epoch_seg(model, loader, optimizer=None, device='cpu', threshold=0.5):
    train_mode = optimizer is not None
    model.train(train_mode)

    total_loss = 0.0
    cy_hits, cy_total = 0, 0
    ve_hits, ve_total = 0, 0

    for batch in loader:
        x          = batch['x'].to(device)
        y_seg      = batch['y_seg'].to(device)       # (B, N_FIXED, H, W)
        cy_true_d  = batch['cy_true_d'].cpu().numpy()
        cy_true_r  = batch['cy_true_r'].cpu().numpy()
        ve_true_d  = batch['ve_true_d'].cpu().numpy()
        ve_true_r  = batch['ve_true_r'].cpu().numpy()
        valid_cy   = batch['valid_cy'].cpu().numpy()
        valid_ve   = batch['valid_ve'].cpu().numpy()

        if train_mode:
            optimizer.zero_grad()

        logits = model(x)                            # (B, 3, N_FIXED, H, W)
        loss   = compute_seg_loss(logits, y_seg)

        if train_mode:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item() * x.size(0)

        for i in range(x.size(0)):
            cy_dets, ve_dets = decode_softmax_detections(
                logits[i:i+1].detach().cpu(), threshold=threshold)
            if valid_cy[i]:
                cy_total += 1
                # いずれかの検出が真値に十分近ければヒット
                if any(abs(det[1]-cy_true_d[i]) <= D_TOL and abs(det[2]-cy_true_r[i]) <= R_TOL
                       for det in cy_dets):
                    cy_hits += 1
            if valid_ve[i]:
                ve_total += 1
                if any(abs(det[1]-ve_true_d[i]) <= D_TOL and abs(det[2]-ve_true_r[i]) <= R_TOL
                       for det in ve_dets):
                    ve_hits += 1

    avg_loss    = total_loss / max(len(loader.dataset), 1)
    cy_hit_rate = cy_hits / max(cy_total, 1)
    ve_hit_rate = ve_hits / max(ve_total, 1)
    return avg_loss, cy_hit_rate, ve_hit_rate


In [12]:
# ===== RadarUNet3DSoftmax 学習 =====
# MODEL_PATH_SEG は split セル（428f4187）で GENERALIZATION_EXPERIMENT フラグにより決定済み

train_ds_seg = MultiTargetDatasetSeg(train_df)
val_ds_seg   = MultiTargetDatasetSeg(val_df)
test_ds_seg  = MultiTargetDatasetSeg(test_df)

train_loader_seg = DataLoader(train_ds_seg, batch_size=BATCH_SIZE, shuffle=True)
val_loader_seg   = DataLoader(val_ds_seg,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_seg  = DataLoader(test_ds_seg,  batch_size=BATCH_SIZE, shuffle=False)

model_seg     = RadarUNet3DSoftmax(n_angles=N_FIXED).to(DEVICE)
optimizer_seg = torch.optim.Adam(model_seg.parameters(), lr=LR)

best_val_seg  = float("inf")
history_seg   = []

for epoch in range(EPOCHS):
    train_loss, train_cy, train_ve = run_epoch_seg(model_seg, train_loader_seg, optimizer_seg, DEVICE)
    val_loss,   val_cy,   val_ve   = run_epoch_seg(model_seg, val_loader_seg,   None,          DEVICE)

    history_seg.append({
        "epoch": epoch + 1,
        "train_loss": train_loss, "train_cy_hit": train_cy, "train_ve_hit": train_ve,
        "val_loss":   val_loss,   "val_cy_hit":   val_cy,   "val_ve_hit":   val_ve,
    })

    if val_loss < best_val_seg:
        best_val_seg = val_loss
        torch.save(model_seg.state_dict(), MODEL_PATH_SEG)

    print(f"epoch={epoch+1:02d}  train={train_loss:.4f} cy={train_cy:.3f} ve={train_ve:.3f}"
          f"  |  val={val_loss:.4f} cy={val_cy:.3f} ve={val_ve:.3f}")

print(f"best model saved to: {MODEL_PATH_SEG}")
hist_seg_df = pd.DataFrame(history_seg)
display(hist_seg_df.tail())

データ読み込み中... (280 サンプル)
読み込み完了
データ読み込み中... (60 サンプル)
読み込み完了
データ読み込み中... (100 サンプル)
読み込み完了
epoch=01  train=0.1903 cy=0.000 ve=0.000  |  val=0.0139 cy=0.000 ve=0.000
epoch=02  train=0.0088 cy=0.000 ve=0.000  |  val=0.0042 cy=0.000 ve=0.000
epoch=03  train=0.0035 cy=0.000 ve=0.142  |  val=0.0022 cy=0.000 ve=0.382
epoch=04  train=0.0026 cy=0.000 ve=0.179  |  val=0.0020 cy=0.000 ve=0.294
epoch=05  train=0.0020 cy=0.137 ve=0.358  |  val=0.0013 cy=0.846 ve=0.559
epoch=06  train=0.0017 cy=0.795 ve=0.366  |  val=0.0016 cy=0.846 ve=0.147
epoch=07  train=0.0017 cy=0.863 ve=0.351  |  val=0.0012 cy=1.000 ve=0.206
epoch=08  train=0.0011 cy=0.904 ve=0.545  |  val=0.0006 cy=1.000 ve=0.471
epoch=09  train=0.0009 cy=0.945 ve=0.507  |  val=0.0004 cy=1.000 ve=0.529
epoch=10  train=0.0010 cy=0.849 ve=0.410  |  val=0.0004 cy=1.000 ve=0.559
epoch=11  train=0.0009 cy=0.822 ve=0.530  |  val=0.0006 cy=1.000 ve=0.647
epoch=12  train=0.0005 cy=0.959 ve=0.560  |  val=0.0003 cy=1.000 ve=0.588
epoch=13  train=0.0006 

,epoch,train_loss,train_cy_hit,train_ve_hit,val_loss,val_cy_hit,val_ve_hit
25,26,0.000368,0.965753,0.694030,0.000358,1.0,0.352941
26,27,0.000254,0.986301,0.768657,0.000151,1.0,1.000000
27,28,0.000193,1.000000,0.940299,0.000113,1.0,1.000000
28,29,0.000183,1.000000,0.902985,0.000102,1.0,1.000000
29,30,0.000183,0.993151,0.977612,0.000107,1.0,1.000000


In [ ]:
# ===== RadarUNet3DSoftmax シナリオテスト =====

SCENARIO_META_CSV = "./learn_dataset_scenario_test/metadata.csv"
scenario_df = pd.read_csv(SCENARIO_META_CSV)
scenario_df = scenario_df[scenario_df["valid_all"] == 1].reset_index(drop=True)
print(f"scenario test: {len(scenario_df)} samples")

best_model_seg = RadarUNet3DSoftmax(n_angles=N_FIXED).to(DEVICE)
best_model_seg.load_state_dict(torch.load(MODEL_PATH_SEG, map_location=DEVICE))
best_model_seg.eval()

rows_sc_seg = []
for i in range(len(scenario_df)):
    row = scenario_df.iloc[i]
    npz_path = row["file"]
    if not os.path.isabs(npz_path):
        npz_path = os.path.normpath(os.path.join(".", npz_path))
    x = load_all_rd_maps(npz_path)
    x_tensor = torch.from_numpy(x).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        logits = best_model_seg(x_tensor)
    cy_dets, ve_dets = decode_softmax_detections(logits.cpu())

    cy_true_d = int(row["cyclist_true_d_idx"]); cy_true_r = int(row["cyclist_true_r_idx"])
    ve_true_d = int(row["vehicle_true_d_idx"]);  ve_true_r = int(row["vehicle_true_r_idx"])

    cy_found = len(cy_dets) > 0
    ve_found = len(ve_dets) > 0
    cy_hit   = any(abs(det[1]-cy_true_d) <= D_TOL and abs(det[2]-cy_true_r) <= R_TOL for det in cy_dets)
    ve_hit   = any(abs(det[1]-ve_true_d) <= D_TOL and abs(det[2]-ve_true_r) <= R_TOL for det in ve_dets)
    n_cy = len(cy_dets)
    n_ve = len(ve_dets)

    rows_sc_seg.append({
        "step": i, "r_diff": abs(cy_true_r - ve_true_r),
        "n_cy": n_cy, "n_ve": n_ve,
        "cy_found": cy_found, "ve_found": ve_found,
        "cy_hit": cy_hit,   "ve_hit": ve_hit,
        "cy_dets": cy_dets, "ve_dets": ve_dets,
    })

sc_seg_df = pd.DataFrame(rows_sc_seg)
total = len(sc_seg_df)

print(f"{'step':>5}  {'r_diff':>6}  {'n_cy':>4}  {'n_ve':>4}  cy_hit  ve_hit")
print("-" * 55)
for _, r in sc_seg_df.iterrows():
    print(f"{int(r.step):>5}  {int(r.r_diff):>6}  {int(r.n_cy):>4}  {int(r.n_ve):>4}"
          f"  {'o' if r.cy_hit else '-':>6}  {'o' if r.ve_hit else '-':>6}")

both_hit = (sc_seg_df["cy_hit"] & sc_seg_df["ve_hit"]).sum()
print(f"=== 位置精度 ===")
print(f"cy_hit: {sc_seg_df.cy_hit.sum():>2}/{total} ({sc_seg_df.cy_hit.mean()*100:.1f}%)")
print(f"ve_hit: {sc_seg_df.ve_hit.sum():>2}/{total} ({sc_seg_df.ve_hit.mean()*100:.1f}%)")
print(f"両方正確: {both_hit:>2}/{total} ({both_hit/total*100:.1f}%)")
print(f"=== 検出数（平均）===")
print(f"cy: {sc_seg_df.n_cy.mean():.2f}  ve: {sc_seg_df.n_ve.mean():.2f}")


In [ ]:
# ===== 単一ターゲット テスト（誤検出率の確認）=====
# cy-only: cyclist のみ存在 → cy が検出され、ve が誤検出されないか
# ve-only: vehicle のみ存在 → ve が検出され、cy が誤検出されないか
#
# learn_dataset_single_object を使用（N_FIXED=10で学習データと同フォーマット）
# 学習済みインデックス（先頭340件）を除いた holdout 260件のみテストに使う

single_all_df = pd.read_csv(TRAIN_META_CSV)
single_all_df = single_all_df[single_all_df["valid_all"] == 1].reset_index(drop=True)
single_all_df = single_all_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
# 学習・valに使った先頭340件を除外したholdout
single_holdout_df = single_all_df.iloc[340:].reset_index(drop=True)

cy_only_holdout = single_holdout_df[
    single_holdout_df["valid_cyclist"].astype(str).str.strip().isin(["1", "True"])
    & ~single_holdout_df["valid_vehicle"].astype(str).str.strip().isin(["1", "True"])
].reset_index(drop=True)

ve_only_holdout = single_holdout_df[
    single_holdout_df["valid_vehicle"].astype(str).str.strip().isin(["1", "True"])
    & ~single_holdout_df["valid_cyclist"].astype(str).str.strip().isin(["1", "True"])
].reset_index(drop=True)

print(f"holdout: cy-only={len(cy_only_holdout)}, ve-only={len(ve_only_holdout)}")


def run_single_target_test_v2(sub_df, target_class):
    """
    target_class: 'cy' or 've'
    returns: DataFrame, hit数, FP数, 総サンプル数
    """
    true_d_col = "cyclist_true_d_idx" if target_class == "cy" else "vehicle_true_d_idx"
    true_r_col = "cyclist_true_r_idx" if target_class == "cy" else "vehicle_true_r_idx"

    rows = []
    for i in range(len(sub_df)):
        row = sub_df.iloc[i]
        npz_path = row["file"]
        if not os.path.isabs(npz_path):
            npz_path = os.path.normpath(os.path.join(".", npz_path))
        x = load_all_rd_maps(npz_path)
        x_tensor = torch.from_numpy(x).unsqueeze(0).float().to(DEVICE)
        with torch.no_grad():
            logits = best_model_seg(x_tensor)
        cy_dets, ve_dets = decode_softmax_detections(logits.cpu())

        true_d = int(row[true_d_col])
        true_r = int(row[true_r_col])

        target_dets   = cy_dets if target_class == "cy" else ve_dets
        opponent_dets = ve_dets if target_class == "cy" else cy_dets

        found = len(target_dets) > 0
        hit   = any(abs(det[1] - true_d) <= D_TOL and abs(det[2] - true_r) <= R_TOL
                    for det in target_dets)
        # 存在しないクラスが閾値を超えて検出された場合を誤検出とする
        fp    = len(opponent_dets) > 0

        rows.append({"i": i, "found": found, "hit": hit, "fp": fp,
                     "true_d": true_d, "true_r": true_r,
                     "target_dets": target_dets, "fp_dets": opponent_dets})

    result_df = pd.DataFrame(rows)
    hits     = int(result_df["hit"].sum())
    fp_count = int(result_df["fp"].sum())
    total    = len(result_df)
    return result_df, hits, fp_count, total


print()
print("=" * 50)
print("cy-only シナリオ（cyclist のみ存在）")
print("=" * 50)
cy_only_df, cy_hits, cy_fp, cy_total = run_single_target_test_v2(cy_only_holdout, "cy")
print(f"サンプル数             : {cy_total}")
print(f"cyclist 検出率  (hit)  : {cy_hits}/{cy_total} ({cy_hits/max(cy_total,1)*100:.1f}%)")
print(f"vehicle 誤検出率 (FP)  : {cy_fp}/{cy_total}  ({cy_fp/max(cy_total,1)*100:.1f}%)")

print()
print("=" * 50)
print("ve-only シナリオ（vehicle のみ存在）")
print("=" * 50)
ve_only_df, ve_hits, ve_fp, ve_total = run_single_target_test_v2(ve_only_holdout, "ve")
print(f"サンプル数             : {ve_total}")
print(f"vehicle 検出率  (hit)  : {ve_hits}/{ve_total} ({ve_hits/max(ve_total,1)*100:.1f}%)")
print(f"cyclist 誤検出率 (FP)  : {ve_fp}/{ve_total}  ({ve_fp/max(ve_total,1)*100:.1f}%)")


In [ ]:
# ===== 2物体 holdout テスト（test_df, 100件）=====
# fixed_angle データセットの holdout（学習・val 未使用）
# scenario_test より母数が多くランダムな r_diff 分布で汎化性能を評価する

print(f"2物体 holdout test: {len(test_df)} samples")

rows_test = []
for i in range(len(test_df)):
    row = test_df.iloc[i]
    npz_path = row["file"]
    if not os.path.isabs(npz_path):
        npz_path = os.path.normpath(os.path.join(".", npz_path))
    x = load_all_rd_maps(npz_path)
    x_tensor = torch.from_numpy(x).unsqueeze(0).float().to(DEVICE)
    with torch.no_grad():
        logits = best_model_seg(x_tensor)
    cy_dets, ve_dets = decode_softmax_detections(logits.cpu())

    cy_true_d = int(row["cyclist_true_d_idx"]); cy_true_r = int(row["cyclist_true_r_idx"])
    ve_true_d = int(row["vehicle_true_d_idx"]);  ve_true_r = int(row["vehicle_true_r_idx"])
    r_diff = abs(cy_true_r - ve_true_r)

    cy_found = len(cy_dets) > 0
    ve_found = len(ve_dets) > 0
    cy_hit   = any(abs(det[1]-cy_true_d) <= D_TOL and abs(det[2]-cy_true_r) <= R_TOL for det in cy_dets)
    ve_hit   = any(abs(det[1]-ve_true_d) <= D_TOL and abs(det[2]-ve_true_r) <= R_TOL for det in ve_dets)
    n_cy = len(cy_dets)
    n_ve = len(ve_dets)

    rows_test.append({
        "step": i, "r_diff": r_diff,
        "n_cy": n_cy, "n_ve": n_ve,
        "cy_found": cy_found, "ve_found": ve_found,
        "cy_hit": cy_hit, "ve_hit": ve_hit,
    })

test_result_df = pd.DataFrame(rows_test)
total = len(test_result_df)

# --- per-sample 出力 ---
print(f"{'step':>5}  {'r_diff':>6}  {'n_cy':>4}  {'n_ve':>4}  cy_hit  ve_hit")
print("-" * 55)
for _, r in test_result_df.iterrows():
    print(f"{int(r.step):>5}  {int(r.r_diff):>6}  {int(r.n_cy):>4}  {int(r.n_ve):>4}"
          f"  {'o' if r.cy_hit else '-':>6}  {'o' if r.ve_hit else '-':>6}")

# --- 全体サマリ ---
both_hit = (test_result_df["cy_hit"] & test_result_df["ve_hit"]).sum()
print(f"\n=== 位置精度 ===")
print(f"cy_hit   : {test_result_df.cy_hit.sum():>3}/{total} ({test_result_df.cy_hit.mean()*100:.1f}%)")
print(f"ve_hit   : {test_result_df.ve_hit.sum():>3}/{total} ({test_result_df.ve_hit.mean()*100:.1f}%)")
print(f"両方正確 : {both_hit:>3}/{total} ({both_hit/total*100:.1f}%)")
print(f"\n=== 検出数（平均）===")
print(f"cy: {test_result_df.n_cy.mean():.2f}  ve: {test_result_df.n_ve.mean():.2f}")

# --- r_diff 別の集計 ---
print(f"\n=== r_diff 別の ve_hit（近接時の精度）===")
bins = [(0, 0), (1, 2), (3, 5), (6, 999)]
labels = ["r_diff=0", "r_diff=1-2", "r_diff=3-5", "r_diff≥6"]
for (lo, hi), label in zip(bins, labels):
    sub = test_result_df[(test_result_df["r_diff"] >= lo) & (test_result_df["r_diff"] <= hi)]
    if len(sub) == 0:
        continue
    vh = sub["ve_hit"].sum()
    print(f"  {label:>12}: ve_hit {vh}/{len(sub)} ({vh/len(sub)*100:.1f}%)"
          f"  cy_hit {sub['cy_hit'].sum()}/{len(sub)} ({sub['cy_hit'].mean()*100:.1f}%)")
